# EDA Datavidia 11.0 Task 1: status penanganan pengiriman

## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency
from IPython.display import display, HTML

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

BASE = "dataset/Task-1-Datavidia-11.0/"
TARGET = "status_penanganan"

In [ ]:
shipments = pd.read_csv(BASE + "train/shipments.csv")
courier   = pd.read_csv(BASE + "train/couriers.csv")
custom    = pd.read_csv(BASE + "train/customers.csv")
hubs      = pd.read_csv(BASE + "train/hubs.csv")
invoices  = pd.read_csv(BASE + "train/invoices.csv")
promo     = pd.read_csv(BASE + "train/promotions.csv")
route     = pd.read_csv(BASE + "train/route_segments.csv")
event     = pd.read_csv(BASE + "train/scan_events.csv",
                        dtype={"event_type": "category", "device_type": "category"})
service   = pd.read_csv(BASE + "train/service_packages.csv")

kolom_asli = list(shipments.columns)

In [ ]:
def show(df, judul=None):
    if judul:
        display(HTML(f"<h4>{judul}</h4>"))
    display(HTML(df.to_html(border=1)))


def eta_sq(df, cols):
    g = df.groupby(TARGET)[cols]
    n, mu = g.count(), g.mean()
    grand = df[cols].mean()
    ssb = (n * (mu - grand) ** 2).sum()
    sst = ((df[cols] - grand) ** 2).sum()
    return (ssb / sst).sort_values(ascending=False)


def cramers_v(x, y):
    ct = pd.crosstab(x, y)
    chi2 = chi2_contingency(ct, correction=False)[0]
    return np.sqrt(chi2 / (ct.values.sum() * (min(ct.shape) - 1)))


def cat_report(df, col, urut="n", maks=15, min_n=1000):
    ct = pd.crosstab(df[col], df[TARGET], normalize="index")
    n = df[col].value_counts().rename("n")
    tab = pd.concat([n, ct], axis=1)
    tab = tab.sort_values("n", ascending=False) if urut == "n" else tab.sort_index()
    fmt = tab.head(maks).copy()
    fmt["n"] = fmt["n"].map("{:,.0f}".format)
    for k in ct.columns:
        fmt[k] = fmt[k].map("{:.1%}".format)
    show(fmt, f"{col} (V = {cramers_v(df[col], df[TARGET]):.4f})")
    base = df[TARGET].value_counts(normalize=True)
    ok = n[n >= min_n].index
    if len(ok):
        lift = (ct / base).loc[ok]
        show(pd.DataFrame({"lift_maks": lift.max().round(2), "di_level": lift.idxmax()}), f"lift {col}")


def bin_report(df, col, q=10):
    tmp = pd.DataFrame({col: pd.qcut(df[col], q, duplicates="drop"), TARGET: df[TARGET]})
    cat_report(tmp, col, urut="index", maks=q)


def signal_map(df, cols, q=10, min_n=5000):
    base = df[TARGET].value_counts(normalize=True)
    rows = []
    for col in cols:
        x = df[col]
        if pd.api.types.is_numeric_dtype(x) and x.nunique() > 30:
            x = pd.qcut(x, q, duplicates="drop")
        ct = pd.crosstab(x, df[TARGET], normalize="index")
        n = x.value_counts()
        ct = ct.loc[n[n >= min_n].index]
        if len(ct) < 2:
            continue
        lift = (ct / base).clip(lower=0.01)
        for k in lift.columns:
            rows.append((col, k, lift[k].max(), lift[k].min()))
    out = pd.DataFrame(rows, columns=["fitur", "kelas", "lift_maks", "lift_min"])
    out["rasio"] = out.lift_maks / out.lift_min
    return out.sort_values("rasio", ascending=False).round(2).reset_index(drop=True)


def ringkas(cols, k=3):
    sm = signal_map(df, cols)
    top = sm.sort_values(["kelas", "rasio"], ascending=[True, False]).groupby("kelas").head(k)
    show(top.reset_index(drop=True), "fitur terkuat per kelas")

## 1. Ringkasan dataset

In [ ]:
datasets = {"courier": courier, "customers": custom, "hubs": hubs, "invoices": invoices,
            "promotions": promo, "route_segments": route, "scan_events": event,
            "service_packages": service, "shipments": shipments}
ringkasan = pd.DataFrame({
    "baris": {k: len(v) for k, v in datasets.items()},
    "kolom": {k: v.shape[1] for k, v in datasets.items()},
    "missing": {k: int(v.isna().sum().sum()) for k, v in datasets.items()},
})
show(ringkasan)

Kesimpulan:

`volume_l` (shipments, 3,0%), `courier_score` (1,1%), `credit_score` (2,0%), `discount` (invoice, 42,4%) dan `durasi_jam` (rute, 2,0%). Dengan 1,24 juta pengiriman, rata-rata tiap pengiriman punya sekitar 5,5 scan, 1,8 segmen rute dan 1,2 invoice, jadi tabel anak harus diringkas dulu per pengiriman sebelum dipakai.

### Visualisasi Section 1: Ringkasan dataset

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
baris = ringkasan["baris"].sort_values()
y = np.arange(len(baris))
axes[0].scatter(baris.values, y, s=70, color="steelblue")
axes[0].set_xscale("log")
axes[0].set_yticks(y, baris.index)
axes[0].set_title("Jumlah baris tiap dataset (skala log)")
axes[0].set_xlabel("Jumlah baris")
axes[0].grid(axis="x", alpha=0.25)
missing_pct = (ringkasan["missing"] / ringkasan["baris"] * 100).sort_values()
im = axes[1].imshow(missing_pct.to_numpy().reshape(1, -1), aspect="auto", cmap="Reds", vmin=0)
axes[1].set_yticks([0], ["Missing"])
axes[1].set_xticks(range(len(missing_pct)), missing_pct.index, rotation=45, ha="right")
axes[1].set_title("Persentase nilai hilang tiap dataset")
for x, value in enumerate(missing_pct.values):
    axes[1].text(x, 0, f"{value:.1f}%", ha="center", va="center", fontsize=8,
                 color="white" if value > missing_pct.max() * 0.55 else "black")
fig.colorbar(im, ax=axes[1], label="Missing (%)", fraction=0.046, pad=0.04)
fig.tight_layout()
plt.show()


## 2. Distribusi target

In [ ]:
vc = shipments[TARGET].value_counts()
show(pd.DataFrame({"n": vc, "persen": (100 * vc / len(shipments)).round(2)}))
vc.sort_values().plot.barh(logx=True, title="Distribusi kelas (skala log)")
plt.show()

Kesimpulan:

normal 38,1%, konfirmasi_alamat 21,9%, tertunda_operasional 14,9%, gagal_antar_awal 10,0%, retur_pengirim 7,1%, rusak_perjalanan 5,0%, klaim_hilang 3,0%. Rasio kelas terbesar terhadap terkecil 12,6:1 dan kelas terkecil masih punya 37.656 baris, jadi ketidakseimbangannya sedang dan tidak perlu resampling. Menebak semua normal hanya benar 38%. Karena Macro F1 memberi bobot sama ke tiap kelas, validasi dibuat stratified dan bobot atau ambang per kelas dituning di akhir.

### Visualisasi Section 2: Distribusi target

In [ ]:
target_counts = shipments[TARGET].value_counts()
labels = [f"{kelas} ({jumlah / len(shipments):.1%})" for kelas, jumlah in target_counts.items()]
fig, ax = plt.subplots(figsize=(9, 7))
wedges, _ = ax.pie(target_counts.values, startangle=90, counterclock=False,
                   wedgeprops={"width": 0.42, "edgecolor": "white"},
                   colors=plt.cm.Set2(np.linspace(0, 1, len(target_counts))))
ax.text(0, 0, f"Total\n{len(shipments):,}", ha="center", va="center", fontsize=13)
ax.legend(wedges, labels, title="Status penanganan", loc="center left",
          bbox_to_anchor=(1, 0.5), frameon=False)
ax.set_title("Komposisi status penanganan pengiriman")
ax.set_aspect("equal")
fig.tight_layout()
plt.show()


## 3. Kualitas data shipments

In [ ]:
kosong = 100 * shipments.isna().mean()
print(kosong[kosong > 0].round(3))
print("duplikat shipment_id:", shipments.shipment_id.duplicated().sum())
show(pd.crosstab(shipments.promo_id.eq("TANPA_PROMO"), shipments.is_promo), "TANPA_PROMO vs is_promo")
show((100 * shipments.groupby(TARGET)["volume_l"].apply(lambda s: s.isna().mean())).round(3).to_frame("volume_l kosong (%)"))

Kesimpulan

Hanya `volume_l` yang kosong (2,98%) dan tidak ada duplikat. Kosongnya tersebar merata di semua kelas (2,89% sampai 3,03%), jadi bukan sinyal dan cukup dibiarkan NaN untuk model berbasis pohon. `is_promo` sama persis dengan `promo_id != "TANPA_PROMO"` (17% pengiriman berpromo), jadi cukup dipakai salah satu.

### Visualisasi Section 3: Kualitas data shipments

In [ ]:

# Menghitung persentase missing value volume_l
missing_volume = kosong["volume_l"]

# Menghitung konsistensi promo_id dengan is_promo
valid_promo = shipments.dropna(subset=["promo_id", "is_promo"])

expected_promo = valid_promo["promo_id"].ne("TANPA_PROMO")
promo_consistency = (
    expected_promo == valid_promo["is_promo"].astype(bool)
).mean() * 100

# Data visualisasi
labels = [
    "Missing Value (volume_l)",
    "Konsistensi Promo"
]

values = [
    missing_volume,
    promo_consistency
]

# Membuat satu bar chart
fig, ax = plt.subplots(figsize=(10, 4))

bars = ax.barh(
    labels,
    values,
    color=["lightcoral", "steelblue"],
    height=0.5
)

# Menampilkan nilai persentase pada setiap bar
for bar, value in zip(bars, values):
    ax.text(
        bar.get_width() + 1,
        bar.get_y() + bar.get_height() / 2,
        f"{value:.2f}%",
        va="center",
        fontsize=10
    )

ax.set_title("Ringkasan Kualitas Data", fontsize=13)
ax.set_xlabel("Persentase (%)")
ax.set_xlim(0, 110)
ax.grid(axis="x", alpha=0.25)
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()


## 4. Shipments: numerik vs target

In [ ]:
num_cols = shipments[kolom_asli].select_dtypes("number").columns.tolist()
eta = eta_sq(shipments, num_cols)
show(eta.round(5).to_frame("eta2"), "Eta kuadrat")
show(shipments.groupby(TARGET)[eta.index[:12].tolist()].mean().T.round(2), "Rata-rata per kelas, 12 teratas")

In [ ]:
anon = ["skor_prioritas_kirim", "indeks_rute_sekunder", "faktor_muatan_internal",
        "varian_cuaca_rute", "indeks_waktu_proses", "skor_kualitas_cek"]
show(shipments.groupby(TARGET)[anon].agg(["mean", "std"]).round(2).T, "Enam kolom anonim per kelas")

Kesimpulan

Sinyal numerik hanya ada di `jumlah_catatan` (0,064), `declared_value` (0,059), `distance_km` (0,035), `flag_alamat_baru` (0,013) dan `ship_month` (0,006). Sisanya hampir nol. Enam kolom anonim punya mean sekitar 50 dan std sekitar 12 di setiap kelas, jadi kemungkinan besar noise dan dibuang dari baseline, lalu dicek ulang lewat permutation importance.

Eta kuadrat hanya melihat beda rata-rata dan dirata-ratakan lintas kelas. Fitur yang hanya menggeser satu kelas langka (misalnya `is_cod` ke retur_pengirim) tampak lemah di sini padahal berguna, jadi jangan menyaring fitur hanya dari angka ini.

### Visualisasi Section 4: Fitur numerik vs target

In [ ]:
eta_top = eta.head(12)
mean_per_class = shipments.groupby(TARGET)[eta_top.index].mean().T
mean_z = mean_per_class.sub(mean_per_class.mean(axis=1), axis=0).div(
    mean_per_class.std(axis=1).replace(0, np.nan), axis=0
).fillna(0)
fig, axes = plt.subplots(1, 2, figsize=(17, 7), gridspec_kw={"width_ratios": [1.15, 1]})
im = axes[0].imshow(mean_z.values, aspect="auto", cmap="RdBu_r", vmin=-2, vmax=2)
axes[0].set_yticks(range(len(mean_z.index)), mean_z.index)
axes[0].set_xticks(range(len(mean_z.columns)), mean_z.columns, rotation=45, ha="right")
axes[0].set_title("Pola rata-rata fitur per status (z-score per fitur)")
axes[0].set_xlabel("Status penanganan")
fig.colorbar(im, ax=axes[0], label="Z-score relatif", fraction=0.046, pad=0.04)
eta_plot = eta_top.sort_values()
y = np.arange(len(eta_plot))
axes[1].hlines(y, 0, eta_plot.values, color="mediumseagreen", linewidth=2)
axes[1].scatter(eta_plot.values, y, color="darkgreen", s=55, zorder=3)
axes[1].set_yticks(y, eta_plot.index)
axes[1].set_title("Kekuatan hubungan fitur numerik")
axes[1].set_xlabel("Eta-kuadrat")
axes[1].grid(axis="x", alpha=0.25)
fig.tight_layout()
plt.show()


## 5. Shipments: kategorikal vs target

In [ ]:
cat_cols = ["channel", "pickup_type", "goods_category", "dest_region", "is_cod", "is_promo",
            "insurance_opt", "gift_wrap", "fragility_flag", "flag_alamat_baru",
            "prioritas_pengiriman", "jumlah_catatan", "qty_items", "coupon_count",
            "ship_month", "ship_dow", "ship_hour"]
vs = pd.Series({c: cramers_v(shipments[c], shipments[TARGET]) for c in cat_cols}).sort_values(ascending=False)
show(vs.round(4).to_frame("cramers_v"))

In [ ]:
for c in ["jumlah_catatan", "flag_alamat_baru", "ship_month", "is_cod"]:
    cat_report(shipments, c, urut="index" if c == "ship_month" else "n")

Kesimpulan

- `jumlah_catatan`: efeknya biner. Tanpa catatan, konfirmasi_alamat 12,6%; satu catatan atau lebih sekitar 35%, dan catatan tambahan tidak menambah apa-apa. Lift 16,5 untuk klaim_hilang di level 7 catatan hanya artefak dari 2 baris.
- `flag_alamat_baru`: konfirmasi_alamat naik dari 20,3% ke 33,2%.
- `ship_month`: bulan 3, 10, 11, 12 membuat tertunda_operasional sekitar 21% (bulan lain sekitar 11%) dan normal turun dari sekitar 41% ke 33%. Kelas lain nyaris tidak bergerak.
- `is_cod`: retur_pengirim naik dari 6,9% ke 9,6%.
- `channel`, `pickup_type`, `dest_region`, `goods_category`, `gift_wrap`, `coupon_count`, `qty_items`, `fragility_flag`, `insurance_opt`, `ship_dow` semuanya V di bawah 0,004, atau noise. `fragility_flag` dan `goods_category` ternyata tidak berhubungan dengan rusak_perjalanan.

### Visualisasi Section 5: Fitur kategorikal vs target

In [ ]:
fitur_tampil = ["is_cod", "flag_alamat_baru", "is_promo", "ship_month"]
baris_heatmap, label_heatmap = [], []
status_urut = shipments[TARGET].dropna().unique()
for fitur in fitur_tampil:
    proporsi = pd.crosstab(shipments[fitur], shipments[TARGET], normalize="index")
    for nilai, row in proporsi.iterrows():
        label_heatmap.append(f"{fitur} = {nilai}")
        baris_heatmap.append(row.reindex(status_urut, fill_value=0))
heatmap = pd.DataFrame(baris_heatmap, index=label_heatmap)
fig, ax = plt.subplots(figsize=(13, 9))
im = ax.imshow(heatmap.values * 100, aspect="auto", cmap="YlGnBu", vmin=0, vmax=60)
ax.set_yticks(range(len(heatmap.index)), heatmap.index)
ax.set_xticks(range(len(heatmap.columns)), heatmap.columns, rotation=35, ha="right")
ax.set_title("Komposisi status menurut kategori terpilih")
ax.set_xlabel("Status penanganan")
ax.set_ylabel("Fitur dan level kategori")
fig.colorbar(im, ax=ax, label="Dalam kategori (%)", fraction=0.035, pad=0.02)
fig.tight_layout()
plt.show()


## 6. Shipments: pendalaman fitur kunci

In [ ]:
shipments["off_hours"] = ((shipments.ship_hour < 6) | (shipments.ship_hour >= 20)).astype("int8")
shipments["has_note"] = (shipments.jumlah_catatan > 0).astype("int8")
shipments["peak_month"] = shipments.ship_month.isin([3, 10, 11, 12]).astype("int8")
shipments["log_declared"] = np.log1p(shipments.declared_value)

### 6.1 Jam pengiriman

In [ ]:
cat_report(shipments, "ship_hour", urut="index", maks=24)
cat_report(shipments, "off_hours")

Kesimpulan

Ada ambang tajam di jam 6 dan jam 20, dan di dalam tiap jendela nilainya datar. Pada jam 6 sampai 19: gagal_antar_awal 9,6%, konfirmasi_alamat 21,3%, retur_pengirim 6,9%, normal 38,8%. Pada jam 20 sampai 5: 14,0%, 27,7%, 8,3% dan 31,2%. Hanya 9,6% pengiriman jatuh di jendela ini. Jadi jam cukup diwakili satu fitur biner `off_hours`. Efeknya menggeser kelas yang berkaitan dengan sisi pelanggan (gagal, konfirmasi, retur) naik, sedangkan normal, rusak, klaim dan tertunda turun.

### 6.2 Nilai barang dan jarak

In [ ]:
bin_report(shipments, "declared_value")
bin_report(shipments, "distance_km")
show(shipments.groupby(TARGET)[["declared_value", "distance_km"]].median().round(0), "Median per kelas")

Kesimpulan

`declared_value` menggerakkan klaim_hilang secara monoton: 0,42% di desil terendah naik ke 13,24% di desil tertinggi (lift 4,37), dengan kenaikan makin curam di ujung atas. Ini bukan ambang dan bukan efek outlier. Meski begitu, bahkan di desil teratas hanya 13% yang berakhir klaim, jadi nilai barang saja tidak cukup.

`distance_km` menggerakkan rusak_perjalanan hampir linear: 0,56% di desil terdekat sampai 11,63% di desil terjauh (lift 2,33). tertunda_operasional ikut naik dari 10,7% ke 18,9% dan normal turun dari 47% ke 31%. Sinyal rusak dari jarak ternyata kuat, bukan lemah seperti dugaan awal.

### 6.3 Sinyal alamat

In [ ]:
show(pd.crosstab([shipments.has_note, shipments.flag_alamat_baru], shipments[TARGET], normalize="index").round(3),
     "has_note dan flag_alamat_baru")

Kesimpulan

Proporsi konfirmasi_alamat: 11,7% tanpa keduanya, 19,8% hanya flag, 33,1% hanya catatan, 46,5% dengan keduanya. Dalam log-odds efeknya +0,62 dan +1,32, jumlahnya 1,94 hampir sama dengan efek gabungan 1,88, jadi dua sinyal ini saling menambah tanpa interaksi berarti dan model tidak perlu fitur gabungan. Puncaknya hanya 46,5%, sehingga sinyal alamat dari tabel lain (misalnya `address_quality` pelanggan) masih perlu dicari.

### 6.4 Fitur turunan yang tidak berguna

In [ ]:
cek = pd.DataFrame({
    "same_city": (shipments.kota_asal == shipments.kota_tujuan).astype("int8"),
    "order_gt24": (shipments.order_to_ship_hours > 24).astype("int8"),
    TARGET: shipments[TARGET],
})
cat_report(cek, "same_city")
cat_report(cek, "order_gt24")
vol_ratio = shipments.berat_volumetrik / shipments.weight_kg
show(vol_ratio.groupby(shipments[TARGET]).describe().round(3), "berat_volumetrik / weight_kg")

Kesimpulan

`same_city` tidak membedakan kelas (V 0,002). `order_gt24` hanya sedikit menaikkan tertunda_operasional (16,3% vs 14,9%). Rasio `berat_volumetrik / weight_kg` berdistribusi sama di semua kelas, jadi `berat_volumetrik` tidak menambah informasi di luar `weight_kg`. Ketiganya tidak dipakai.

### Visualisasi Section 6: Pendalaman fitur kunci

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(16, 16))
jam_status = pd.crosstab(shipments["ship_hour"], shipments[TARGET], normalize="index").reindex(range(24))
for status in jam_status.columns:
    axes[0, 0].plot(jam_status.index, jam_status[status] * 100, marker=".", label=status)
axes[0, 0].set_title("Komposisi status menurut jam pengiriman")
axes[0, 0].set_xlabel("Jam")
axes[0, 0].set_ylabel("Dalam-jam (%)")
axes[0, 0].set_xticks(range(0, 24, 2))
axes[0, 0].legend(fontsize=7, ncol=2)

for ax, kolom, kelas, judul in [
    (axes[0, 1], "declared_value", "klaim_hilang", "Klaim hilang menurut desil nilai barang"),
    (axes[1, 0], "distance_km", "rusak_perjalanan", "Rusak perjalanan menurut desil jarak"),
]:
    bins = pd.qcut(shipments[kolom], q=10, duplicates="drop")
    proporsi = shipments.assign(_bin=bins).groupby("_bin", observed=True)[TARGET].apply(
        lambda s: s.eq(kelas).mean() * 100
    )
    ax.plot(range(len(proporsi)), proporsi.values, marker="o", color="teal")
    ax.set_xticks(range(len(proporsi)))
    ax.set_xticklabels([f"D{i + 1}" for i in range(len(proporsi))])
    ax.set_title(judul)
    ax.set_xlabel("Desil")
    ax.set_ylabel(f"{kelas} (%)")
    ax.grid(axis="y", alpha=0.25)

alamat = shipments.assign(
    _kombinasi=shipments["has_note"].map({0: "Tanpa catatan", 1: "Ada catatan"})
    + " / " + shipments["flag_alamat_baru"].map({0: "Alamat lama", 1: "Alamat baru"})
)
alamat_status = pd.crosstab(alamat["_kombinasi"], alamat[TARGET], normalize="index") * 100
im = axes[1, 1].imshow(alamat_status.values, aspect="auto", cmap="YlOrRd", vmin=0)
axes[1, 1].set_yticks(range(len(alamat_status.index)), alamat_status.index)
axes[1, 1].set_xticks(range(len(alamat_status.columns)), alamat_status.columns, rotation=45, ha="right")
axes[1, 1].set_title("Komposisi status menurut sinyal alamat")
for row in range(alamat_status.shape[0]):
    for col in range(alamat_status.shape[1]):
        value = alamat_status.iloc[row, col]
        axes[1, 1].text(col, row, f"{value:.0f}", ha="center", va="center", fontsize=7,
                        color="white" if value > alamat_status.values.max() * 0.55 else "black")

# Titik menonjolkan bahwa kedua nilai Cramer's V sangat kecil.
turunan_v = pd.Series({
    "same_city": cramers_v(cek["same_city"], cek[TARGET]),
    "order_gt24": cramers_v(cek["order_gt24"], cek[TARGET]),
}).sort_values()
y = np.arange(len(turunan_v))
axes[2, 0].hlines(y, 0, turunan_v.values, color="silver", linewidth=2)
axes[2, 0].scatter(turunan_v.values, y, color="slategray", s=60, zorder=3)
axes[2, 0].set_yticks(y, turunan_v.index)
axes[2, 0].set_title("Kekuatan sinyal fitur turunan")
axes[2, 0].set_xlabel("Cramer's V")
axes[2, 0].grid(axis="x", alpha=0.25)

rasio_plot = (shipments["berat_volumetrik"] / shipments["weight_kg"]).replace([np.inf, -np.inf], np.nan)
kelas_urut = shipments[TARGET].value_counts().index.tolist()
data_box = [
    rasio_plot.loc[shipments[TARGET].eq(kelas)].dropna().sample(
        n=min(2500, rasio_plot.loc[shipments[TARGET].eq(kelas)].notna().sum()), random_state=42
    ).values
    for kelas in kelas_urut
]
axes[2, 1].boxplot(data_box, tick_labels=kelas_urut, showfliers=False)
axes[2, 1].set_title("Rasio berat volumetrik / berat aktual")
axes[2, 1].set_ylabel("Rasio")
axes[2, 1].tick_params(axis="x", labelrotation=35)
fig.tight_layout()
plt.show()


## 7. Tabel anak: scan, rute, invoice

### 7.1 Agregasi per pengiriman

In [ ]:
idx = pd.Index(shipments.shipment_id)
for d in (event, route, invoices):
    d["sid"] = idx.get_indexer(d.shipment_id).astype("int32")
    print(len(d), "baris tanpa induk:", (d.sid < 0).sum())

In [ ]:
g = event.groupby("sid")
ev = g.agg(n_scan=("gap_hours", "size"), gap_sum=("gap_hours", "sum"),
           gap_mean=("gap_hours", "mean"), gap_max=("gap_hours", "max"),
           manual_ratio=("is_manual", "mean"), q_mean=("scan_quality", "mean"),
           q_min=("scan_quality", "min"),
           first_event=("event_type", "first"), last_event=("event_type", "last"))
ev_cnt = (event.groupby(["sid", "event_type"], observed=True).size()
          .unstack(fill_value=0).add_prefix("ev_"))
ev = ev.join(ev_cnt)

route["dur_kosong"] = route.durasi_jam.isna().astype("int8")
rt = route.groupby("sid").agg(n_seg=("urutan", "size"), jarak_seg=("jarak_segmen_km", "sum"),
                              durasi_seg=("durasi_jam", "sum"), dur_kosong=("dur_kosong", "sum"),
                              biaya_seg=("biaya_segmen", "sum"), n_hub_seg=("segmen_hub_id", "nunique"))
rt["kecepatan"] = rt.jarak_seg / rt.durasi_seg.replace(0, np.nan)
moda = route.groupby(["sid", "moda"]).size().unstack(fill_value=0).add_prefix("moda_")
rt = rt.join(moda)
rt["n_moda"] = (moda > 0).sum(axis=1)

invoices["tertunda"] = (invoices.payment_status == "tertunda").astype("int8")
inv = invoices.groupby("sid").agg(n_inv=("amount", "size"), inv_total=("amount", "sum"),
                                  penalti=("is_penalty", "max"), inv_tertunda=("tertunda", "sum"),
                                  hari_bayar_maks=("days_to_pay", "max"),
                                  bulan_inv_maks=("issued_month", "max"))
jenis = invoices.groupby(["sid", "invoice_type"]).size().unstack(fill_value=0).add_prefix("inv_")
inv = inv.join(jenis)

df = shipments.join(ev).join(rt).join(inv).copy()

In [ ]:
show((100 * df[["n_scan", "n_seg", "n_inv"]].isna().groupby(df[TARGET]).mean()).round(2),
     "Pengiriman tanpa baris di tabel anak (%) per kelas")
hitung = (["n_scan", "n_seg", "n_inv", "inv_total", "inv_tertunda", "penalti", "n_moda"]
          + list(ev_cnt.columns) + list(moda.columns) + list(jenis.columns))
df[hitung] = df[hitung].fillna(0)

In [ ]:
#visualisasi distribusi kelas per kelas target
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fitur = ["n_scan", "n_seg", "n_inv"]
judul = ["Jumlah Scan", "Jumlah Segmen Rute", "Jumlah Invoice"]

for ax, col, title in zip(axes, fitur, judul):
    temp = df.groupby(TARGET)[col].mean().sort_values()
    temp.plot(kind="barh", ax=ax)
    ax.set_title(f"Rata-rata {title} per Kelas")
    ax.set_xlabel("Rata-rata")
    ax.set_ylabel("Status Penanganan")

plt.tight_layout()
plt.show()

Data scan events, rute, dan invoice digabung menggunakan `shipment_id ` untuk analisis karakteristik status pengiriman

### 7.2 Scan events

In [ ]:
df["n_upaya"] = df.ev_upaya_antar.clip(upper=6)
df["n_pickup"] = df.ev_pickup.clip(upper=5)
df["ada_selesai"] = (df.ev_selesai > 0).astype("int8")
for c in ["last_event", "first_event", "ada_selesai", "n_upaya", "n_pickup", "n_scan"]:
    cat_report(df, c, urut="index", maks=20)

In [ ]:
kol_scan = ["gap_sum", "gap_mean", "gap_max", "manual_ratio", "q_mean", "q_min"]
show(eta_sq(df, kol_scan).round(5).to_frame("eta2"), "Eta kuadrat fitur scan")
show(df.groupby(TARGET)[kol_scan].mean().T.round(2), "Rata-rata per kelas")
bin_report(df, "gap_max")
bin_report(df, "manual_ratio", q=5)

In [ ]:
#visualisasi jumlah upaya pengantaran
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

pd.crosstab(df.n_upaya, df[TARGET], normalize="index"
            ).plot(kind="bar", stacked=True, ax=axes[0])

axes[0].set_title("Distribusi Kelas per Jumlah Upaya Antar")
axes[0].set_xlabel("Jumlah Upaya Antar")
axes[0].set_ylabel("Proporsi")
axes[0].legend(title="Status Penanganan", bbox_to_anchor=(1.02, 1))

pd.crosstab(df["ada_selesai"], df[TARGET], normalize="index").plot(kind="bar", stacked=True, ax=axes[1])
axes[1].set_title("Distribusi Kelas per Ketersediaan Selesai")
axes[1].set_xlabel("Ada Selesai (1 = Ya, 0 = Tidak)")
axes[1].set_ylabel("Proporsi")

plt.tight_layout(rect=[0, 0.12, 1, 1])
plt.figure(figsize=(10, 5))
df.boxplot(column="gap_max", by=TARGET, showfliers=False, rot=30)
plt.title("Distribusi Gap Maksimum per Kelas")
plt.suptitle("")
plt.xlabel("Status Penanganan")
plt.ylabel("Gap Maksimum (jam)")
plt.show()

### 7.3 Rute

In [ ]:
df["biaya_per_km"] = df.biaya_seg / df.jarak_seg.replace(0, np.nan)
for c in ["n_seg", "n_moda", "dur_kosong"] + list(moda.columns):
    cat_report(df, c, urut="index")
kol_rute = ["jarak_seg", "durasi_seg", "biaya_seg", "kecepatan", "biaya_per_km"]
show(eta_sq(df, kol_rute).round(5).to_frame("eta2"), "Eta kuadrat fitur rute")
show(df.groupby(TARGET)[kol_rute].mean().T.round(2), "Rata-rata per kelas")
bin_report(df, "kecepatan")

In [ ]:
#visualisasi
kolom_rute = ["jarak_seg", "durasi_seg", "kecepatan"]

judul = ["Rata-rata Jarak Rute", "Rata- rata Durasi rute", "Rata-rata Kecepatan"]

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
for ax, col in zip(axes, kolom_rute):
    nilai = df.groupby(TARGET)[col].mean().sort_values()
    nilai.plot(kind="barh", ax=ax)
    ax.set_title(judul[kolom_rute.index(col)])
    ax.set_xlabel(col)
    ax.set_ylabel("Status Penanganan")
plt.tight_layout()
plt.show()

 `rusak_perjalanan`  memiliki rata-rata jarak rute tertinggi, dan  `tertunda_operasional`  memiliki rata-rata durasi tertinggi, juga kecepata rata-rata terendah.

### 7.4 Invoice

In [ ]:
show(pd.crosstab(invoices.invoice_type, invoices.discount.isna()), "invoice_type vs discount kosong")
df["lag_bulan_inv"] = (df.bulan_inv_maks - df.ship_month) % 12
for c in list(jenis.columns) + ["penalti", "inv_tertunda", "n_inv", "lag_bulan_inv"]:
    cat_report(df, c, urut="index")
kol_inv = ["inv_total", "hari_bayar_maks"]
show(eta_sq(df, kol_inv).round(5).to_frame("eta2"), "Eta kuadrat fitur invoice")
bin_report(df, "hari_bayar_maks", q=5)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, col, title in zip(
    axes, ["inv_klaim", "penalti"],
    ["invoice Klaim, ", "invoice Penalti"]):
    pd.crosstab(df[col], df[TARGET], normalize="index").plot(kind="bar", stacked=True, ax=ax)
    ax.set_title(f"Status berdasarkan {col}")
    ax.set_xlabel("1=Ya, 0=Tidak")
    ax.set_ylabel("Proporsi")
    ax.legend(title="Status Penanganan", bbox_to_anchor=(1.02, 1))
plt.tight_layout()
plt.show()

Grafik menunjuukan distribusi status penanganan berdasarkan ada atau tidaknya invoice klaim dan penalti

### 7.5 Pengecekan struktur data

In [ ]:
ada = event.sid.values >= 0
hub_ship = shipments.hub_id.to_numpy()[event.sid.values[ada]]
print("hub di scan sama dengan hub shipment:", (event.hub_id.to_numpy()[ada] == hub_ship).mean())
print("pengiriman dengan lebih dari 1 hub di rute:", (rt.n_hub_seg > 1).mean())

d = (df.jarak_seg - df.distance_km).dropna()
print("selisih jarak rute dan distance_km:")
print(d.abs().describe().round(3))
show((d.abs() > 0.5).groupby(df[TARGET]).mean().round(4).to_frame("selisih > 0,5 km"))

pos = event.groupby("sid").cumcount()
tot = event.sid.map(ev.n_scan)
print("scan selesai yang bukan scan terakhir:", ((event.event_type == "selesai") & (pos < tot - 1)).sum())
print(df.first_event.value_counts(normalize=True).round(3))

In [ ]:
ringkas(["last_event", "first_event", "ada_selesai", "n_upaya", "n_pickup", "n_scan", "gap_max", "gap_mean",
         "manual_ratio", "q_mean", "n_seg", "n_moda", "kecepatan", "biaya_per_km"] +
        list(jenis.columns) + ["penalti", "inv_tertunda", "n_inv", "lag_bulan_inv", "hari_bayar_maks"])

In [ ]:


#presentase kecocokan hub scan, rute satu hub, scan pertama pickup, dan scan terakhir selesai
hub_scan = event["hub_id"].to_numpy()
hub_ship = shipments.hub_id.to_numpy()[event.sid.values]
kecocokan_hub_scan = 100 * (hub_scan[ada] == hub_ship[ada]).mean()

rute_satu_hub = 100 * (rt.n_hub_seg == 1).mean()
scan_pertama_pickup = 100 * (df.first_event == "pickup").mean()

cek = pd.Series({ "Kecocokan Hub Scan" : kecocokan_hub_scan,

"Rute Satu Hub": rute_satu_hub,
"Scan Pertama Pickup": scan_pertama_pickup,
})

plt.figure(figsize=(10, 5))
cek.sort_values().plot(kind="barh")
plt.title("Hasil Cek Konsistensi Data")
plt.xlabel("Presentase(%)")
plt.ylabel("Pemeriksaan")
plt.xlim(0, 105)

plt.figure(figsize=(9, 5))
(d.abs()).hist(bins=20)
plt.title("Distribusi Selisih Jarak Rute dan Shipment")
plt.xlabel("Selisih Jarak (km)")
plt.ylabel("Frekuensi")

plt.tight_layout()
plt.show()


Kesimpulan

 `konfirmasi_alamat` memiliki rata-rata upaya antar tertinggi (1.85 kali),  `rusak perjalanan`jarak terjauh (1.174,44 km), dan `tertunda operasional` durasi tertinggi (32.90 jam) juga invoice penalti 56.3%. pemeriksaan juga menunnjukkan konsistensi data yang baik di 100%.

## 8. Tabel master: kurir, pelanggan, hub, layanan, promo

In [ ]:
master = [(courier, "courier_id", "kr_"), (custom, "customer_id", "cu_"), (hubs, "hub_id", "hb_"),
          (service, "package_id", "sv_"), (promo, "promo_id", "pr_")]
for tabel, kunci, pref in master:
    print(pref, "kunci ganda:", tabel[kunci].duplicated().sum())
    t = tabel.set_index(kunci).add_prefix(pref)
    df = df.drop(columns=t.columns, errors="ignore").join(t, on=kunci)
df = df.copy()
print(len(df), len(shipments))
show(df[["kr_vehicle_type", "cu_segment", "hb_hub_type", "sv_speed_class", "pr_tipe_promo"]]
     .isna().mean().round(4).to_frame("kosong"), "Hasil join ke tabel master")

In [ ]:
kat_master = ["kr_vehicle_type", "kr_contract_type", "kr_shift_code", "kr_training_level",
              "kr_sertifikasi_hazardous", "cu_region", "cu_segment", "cu_is_business", "cu_preferred_payment",
              "cu_loyalty_tier", "cu_address_quality", "cu_is_verified", "cu_kota_tipe",
              "cu_metode_notifikasi", "cu_tagihan_langganan", "hb_region", "hb_hub_type",
              "hb_automation_level", "sv_speed_class", "sv_handling_class", "sv_insurance_default",
              "sv_cod_allowed", "pr_tipe_promo", "pr_nama_promo"]
vm = pd.Series({c: cramers_v(df[c], df[TARGET]) for c in kat_master}).sort_values(ascending=False)
show(vm.round(4).to_frame("cramers_v"))
for c in vm.index[:6]:
    cat_report(df, c, maks=10)

In [ ]:
num_master = ["kr_experience_years", "kr_courier_score", "kr_jarak_domisili_hub_km", "kr_kapasitas_muatan_kg",
              "cu_tenure_months", "cu_credit_score", "cu_complaint_index", "hb_capacity_index",
              "hb_density_index", "hb_luas_gudang_m2", "sv_cost_multiplier", "sv_max_weight_kg",
              "sv_fragile_score", "sv_estimasi_hari", "sv_biaya_dasar", "pr_diskon_persen"]
show(eta_sq(df, num_master).round(5).to_frame("eta2"), "Eta kuadrat fitur master numerik")

In [ ]:
df["lewat_kapasitas"] = (df.weight_kg > df.kr_kapasitas_muatan_kg).astype("int8")
df["lewat_batas_paket"] = (df.weight_kg > df.sv_max_weight_kg).astype("int8")
df["cod_dilarang"] = ((df.is_cod == 1) & (df.sv_cod_allowed == 0)).astype("int8")
df["beda_hub_kurir"] = (df.hub_id != df.kr_hub_id).astype("int8")
df["luar_jam_hub"] = ((df.ship_hour < df.hb_jam_buka) | (df.ship_hour >= df.hb_jam_tutup)).astype("int8")

a, b, bln = df.pr_bulan_mulai, df.pr_bulan_akhir, df.ship_month
dalam = np.where(a <= b, (bln >= a) & (bln <= b), (bln >= a) | (bln <= b))
df["promo_aktif"] = np.where(a.isna(), -1, dalam).astype("int8")

for c in ["lewat_kapasitas", "lewat_batas_paket", "cod_dilarang", "beda_hub_kurir", "luar_jam_hub", "promo_aktif"]:
    cat_report(df, c, urut="index")
show(pd.crosstab(df.off_hours, df.luar_jam_hub), "off_hours (jam 6 dan 20) vs luar_jam_hub")
show(pd.crosstab(df.kr_shift_code, df.ship_hour, normalize="index").round(3), "shift kurir vs jam kirim")

In [ ]:
ringkas(kat_master + num_master + ["lewat_kapasitas", "lewat_batas_paket", "cod_dilarang",
                                   "beda_hub_kurir", "luar_jam_hub", "promo_aktif"])

In [ ]:
#visualisasi
plt.figure(figsize=(11, 7))
vm.sort_values().plot.barh()

plt.title("Fitur Cramer's dengan Fitur Kategorikal Master")
plt.xlabel("Cramer's V")
plt.ylabel("Fitur Kategorikal")
plt.tight_layout()
plt.show()

eta_master = eta_sq(df, num_master).sort_values()
plt.figure(figsize=(11, 7))
eta_master.plot(kind="barh")

plt.title("Fitur master Numberik dengan Status Penanganan")
plt.xlabel("Eta-Squared")
plt.ylabel("Fitur")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))
pd.crosstab(df["cu_address_quality"], df[TARGET], normalize="index").plot(kind="bar", stacked=True)
plt.title("Distribusi Kualitas Alamat pelanggan berdasarkan Status Penanganan")
plt.xlabel("Kualitas Alamat")
plt.ylabel("Proporsi")
plt.legend(title="Status Penanganan", bbox_to_anchor=(1.02, 1))
plt.tight_layout()
plt.show()


Kesimpulan
`cu_complaint_index` memiliki hubungan tertinggi sama status penanganan(0.2967) dan `cu_address_quality` memberi sinyal pada `konfirmasi_alamat` (lif 3.50). join tabel master berhasil tanpa duplikasi dengan total 1.243.617 baris. batas operasional yaitu jam 0.600-20.00.

## 9. Efek entitas dan kepadatan

In [ ]:
def overdispersi(kolom):
    ct = pd.crosstab(df[kolom], df[TARGET])
    n = ct.sum(axis=1)
    rate = ct.div(n, axis=0)
    p = ct.sum() / ct.values.sum()
    harap = pd.Series({k: (p[k] * (1 - p[k]) / n).mean() for k in ct.columns})
    return (rate.var() / harap).round(2)

entitas = ["courier_id", "hub_id", "customer_id", "package_id", "kota_asal", "kota_tujuan", "promo_id"]
show(pd.DataFrame({e: overdispersi(e) for e in entitas}).T, "Varians antar entitas dibanding varians binomial")

In [ ]:
df["kirim_hub_bulan"] = df.groupby(["hub_id", "ship_month"])["hub_id"].transform("size")
df["kepadatan"] = pd.qcut(df.kirim_hub_bulan, 3, duplicates="drop")
show(df.groupby("peak_month").kirim_hub_bulan.mean().round(1).to_frame("rata-rata kiriman per hub-bulan"))
show(pd.crosstab([df.peak_month, df.kepadatan], df[TARGET], normalize="index").round(3),
     "Distribusi kelas menurut bulan padat dan kepadatan hub")

In [ ]:
#visualisasi varians antar entitas dibanding varians
hasil_var = pd.DataFrame({e: overdispersi(e) for e in entitas}).T
plt.figure(figsize=(12, 6))
plt.imshow(hasil_var,cmap="YlOrRd", aspect="auto")

plt.xticks(range(len(hasil_var.index)), hasil_var.columns, rotation=30, ha="right")
plt.yticks(range(len(hasil_var.index)), hasil_var.index)

plt.colorbar(label="Rasio Varians")
plt.title("Perbandingan Varians status penanganan antar entitas")
plt.tight_layout()
plt.show()

In [ ]:
#visualisasi distribusi rata-rata kepadatan hub
plt.figure(figsize=(7, 5))
df.groupby("kepadatan").kirim_hub_bulan.mean().plot(kind="bar", color="skyblue")

plt.title("Rata-rata Jumlah Kiriman per Hub per Bulan berdasarkan Kepadatan")
plt.xlabel("kepadatan (1= ya, 0  = tidak)")
plt.ylabel("Rata-rata Kiriman per Hub per Bulan")
plt.xticks(ticks=[0, 1, 2], labels=["Rendah", "Sedang", "Tinggi"], rotation=0)
plt.tight_layout()
plt.show()

#Distribusi status penanganan menurut bulan padat dan kepadatan hub
prop = pd.crosstab([df.peak_month, df.kepadatan], df[TARGET], normalize="index")
prop.plot(kind="bar", stacked=True, figsize=(13, 6))

plt.title("Distribusi Status Penanganan menurut Bulan Padat dan Kepadatan Hub")
plt.xlabel("Bulan Padat dan Kepadatan Hub")
plt.ylabel("Proporsi")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

Kesimpulan

`variansi status` penanganan paling bear terjadi antara-hub terutama pada kelas `tertunda_operasional` dan `klaim hilang`. `package id` juga variansinya tinggis, namun variansi antar-pelanggan lbih kecil.
Rata-rata pengirimannya meningkat 28.2% pada peak month dibandingkan non-peak month. proporsi `tertunda operasional` juga lebih tinggi pada kelompok hub dengan kepadatan tinggi.

## 10. Perbandingan train dan test

In [ ]:
test = pd.read_csv(BASE + "test/shipments.csv")
print(len(test), len(shipments), round(len(test) / (len(test) + len(shipments)), 3))
overlap = {c: test[c].isin(shipments[c]).mean() for c in
           ["customer_id", "courier_id", "hub_id", "package_id", "promo_id", "kota_asal", "kota_tujuan"]}
show(pd.Series(overlap).round(4).to_frame("test yang ada di train"))
bulan = pd.DataFrame({"train": shipments.ship_month.value_counts(normalize=True).sort_index(),
                      "test": test.ship_month.value_counts(normalize=True).sort_index()})
show(bulan.round(4), "Proporsi bulan")
num = ["distance_km", "declared_value", "weight_kg", "ship_hour", "jumlah_catatan", "flag_alamat_baru", "is_cod"]
show(pd.DataFrame({"train": shipments[num].mean(), "test": test[num].mean()}).round(3), "Rata-rata fitur")

In [ ]:
#visualisasi
from numpy import mean


bulan.plot(kind="bar", figsize=(12, 5), width=0.8, color=["#1f77b4", "skyblue"])
plt.title("Perbandingan Distribusi Bulan Train vs Test")
plt.xlabel("Bulan Pengiriman")
plt.ylabel("Proporsi")
plt.xticks(rotation=0)
plt.legend(title="Dataset")

plt.tight_layout()
plt.show()

#perbandingan rata-rata fitur train dan test
num = ["distance_km", "declared_value", "weight_kg", "ship_hour", "jumlah_catatan", "flag_alamat_baru", "is_cod"]
means = pd.DataFrame({"train": shipments[num].mean(), "test": test[num].mean()})
mean.compare = pd.DataFrame({"Train": shipments[num].mean(), "Test": test[num].mean()})

mean.compare.plot(kind="bar", figsize=(12, 5), width=0.8, color=["#1f77b4", "skyblue"])
plt.title("Perbandingan Rata-Rata Fitur Train dan Test")
plt.xlabel("Fitur")
plt.ylabel("Rata-Rata")
plt.xticks(rotation=45)
plt.legend(title="Dataset")
plt.tight_layout()
plt.show()


Kesimpulan

Data train dan test mempunyai distribusi bulan dan rata-rata fitur numerik yang relatif sama. pola split menyerupai pembagian acak, bukan berdasarkan waktu atau entitas.
validasi model sebaiknya mempertahankan kemiripan distribusi data. Fitur `peak_month`dapat digunakan selama informasinya tersedia saat predksi, dan datanya tida leakage.

## 11. Kesimpulan akhir

In [ ]:
kandidat = (["off_hours", "has_note", "flag_alamat_baru", "is_cod", "peak_month", "prioritas_pengiriman",
             "declared_value", "distance_km", "weight_kg", "order_to_ship_hours", "qty_items",
             "insurance_opt", "fragility_flag", "channel", "pickup_type", "goods_category", "dest_region",
             "last_event", "first_event", "ada_selesai", "n_upaya", "n_pickup", "n_scan", "gap_max",
             "gap_mean", "manual_ratio", "q_mean", "n_seg", "n_moda", "kecepatan", "biaya_per_km",
             "penalti", "inv_tertunda", "n_inv", "lag_bulan_inv", "hari_bayar_maks", "kepadatan"]
            + list(jenis.columns) + kat_master + num_master)
kandidat = [c for c in dict.fromkeys(kandidat) if c in df.columns]
sm = signal_map(df, kandidat)
top = sm.sort_values(["kelas", "rasio"], ascending=[True, False]).groupby("kelas").head(5)
show(top.reset_index(drop=True), "Lima fitur terkuat per kelas dari semua tabel")

## Leakage Check

In [ ]:
inv_klaim = invoices.groupby("shipment_id")["invoice_type"] \
                    .apply(lambda s: (s == "klaim").any()).rename("has_klaim")
tmp = shipments[["shipment_id", TARGET]].merge(inv_klaim, on="shipment_id", how="left")
show(pd.crosstab(tmp["has_klaim"].fillna(False), tmp[TARGET], normalize="columns").round(3),
     "proporsi has_klaim per kelas")

In [ ]:
evt_has = event.groupby("shipment_id")["event_type"] \
               .apply(lambda s: set(s))
for e in ["selesai", "upaya_antar", "pickup", "sortasi", "transit"]:
    has = evt_has.apply(lambda x: e in x).rename(f"has_{e}")
    tmp = shipments[["shipment_id", TARGET]].merge(has, on="shipment_id", how="left")
    show(pd.crosstab(tmp[f"has_{e}"].fillna(False), tmp[TARGET], normalize="columns").round(3),
         f"proporsi has_{e} per kelas")

In [ ]:

inv_tert = invoices.groupby("shipment_id")["payment_status"] \
                   .apply(lambda s: (s == "tertunda").any()).rename("has_tertunda")
tmp = shipments[["shipment_id", TARGET]].merge(inv_tert, on="shipment_id", how="left")
show(pd.crosstab(tmp["has_tertunda"].fillna(False), tmp[TARGET], normalize="columns").round(3),
     "proporsi has_tertunda per kelas")

In [ ]:
evt_n = event.assign(v=1).pivot_table(
    index="shipment_id", columns="event_type", values="v", aggfunc="sum", fill_value=0)
evt_n = shipments[["shipment_id", TARGET]].merge(evt_n, on="shipment_id", how="left").fillna(0)
show(evt_n.groupby(TARGET).mean(numeric_only=True).round(2), "rata-rata count event per kelas")



In [ ]:
test_shp = pd.read_csv(BASE + "test/shipments.csv")
for c in ["ship_month", "ship_dow", "dest_region", "channel", "pickup_type", "goods_category"]:
    if c in test_shp.columns:
        a = shipments[c].value_counts(normalize=True).rename("train")
        b = test_shp[c].value_counts(normalize=True).rename("test")
        show(pd.concat([a, b], axis=1).fillna(0).round(4), c)

tr_cust = set(shipments.customer_id)
te_cust = set(test_shp.customer_id)
print(f"customer test tidak ada di train: {len(te_cust-tr_cust):,}/{len(te_cust):,} "
      f"({len(te_cust-tr_cust)/len(te_cust):.1%})")

print("\nshipments columns:", shipments.columns.tolist())